<a href="https://colab.research.google.com/github/prongselk/krillguard/blob/main/excel_to_csv_public.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Data upload and basic preparation

Run the following cells to extract the relevant data from the Excel workbook and clean it up.

In [ ]:
import numpy as np
import pandas as pd
import re

In [ ]:
# url of public copy of excel file
raw_url = "https://raw.githubusercontent.com/prongselk/krillguard/main/KrillGUARD_public.xlsx"

# upload
data = pd.read_excel(raw_url, sheet_name="Raw_Data")

# remove header
data = data.truncate(before=6)

# remove rows with no data - keep rows that have at least one non-missing value in columns 3 and beyond
data = data.loc[data.iloc[:, 2:].notna().any(axis=1)]

In [ ]:
# rename columns and edit so they don't have spaces and are lowercase
data = data.rename(columns=lambda x: x.replace(" ", "_"))
data = data.rename(columns=lambda x: str(x).lower())
data = data.rename(columns={data.columns[14]: "time_of_day"})
data = data.rename(columns={data.columns[12]: "sex"})
data = data.rename(columns={data.columns[7]: "start_depth"})
data = data.rename(columns={data.columns[8]: "end_depth"})

In [ ]:
# remove row for collection with date that does not exist
data = data [ data["date"] != "1979-02-29" ]

In [ ]:
# convert date to datetime format and extract year, month and day
data["date_dt"] = pd.to_datetime(data["date"])
data["year"] = data["date_dt"].dt.year
data["month"] = data["date_dt"].dt.month
data["day"] = data["date_dt"].dt.day

In [ ]:
#add a column for year + check for date errors
def convert_date(date):
    try:
        if str(date).isdigit():
            return pd.to_datetime(int(date), origin='1899-12-30', unit='D')
        elif '-' in str(date):
            return pd.to_datetime(date, format='%Y-%m-%d', errors='coerce')
    except:
        return np.nan
    return np.nan


data['correct_date'] = data['date'].apply(convert_date)
data['year'] = data['correct_date'].dt.year
data['decade'] = data['year'].apply(lambda x: f"{x // 10 * 10}s" if pd.notna(x) else np.nan)
num_na_years = data['year'].isna().sum()
na_year = data[data['year'].isna()]

print("Number of NA years:", num_na_years)

na_year_but_date_present = data[data['year'].isna() & data['date'].notna()]
print(na_year_but_date_present)

In [ ]:
# remove "\xa0 line endings
data["species"] = data["species"].apply(lambda x: str(x).replace(u"\xa0", u""))

In [ ]:
# change NAs that indicate missing non-essential data with "Unknown"
data["species"] = data["species"].fillna("Unknown")
data["genus"] = data["genus"].fillna("Unknown")
data["family"] = data["family"].fillna("Unknown")
data["sex"] = data["sex"].fillna("Unknown")
data["life_stage"] = data["life_stage"].fillna("Unknown")
data["time_of_day"] = data["time_of_day"].fillna("Unknown")

In [ ]:
# drop extra/intermediate columns
data = data.drop("date_dt", axis=1)
data = data.drop("correct_date", axis=1)

### Saving basic .csv with intact Notes

Run the next cell to save a .csv file that will have 'Notes' as one column. Continue further if you want separate columns for notes on different things.

In [ ]:
#to save current version as csv
savecsv = data.to_csv('data.csv', index=False)

### Separating notes into different columns

Run the following cells if you want to have notes that are separated by topic into different columns.

In [ ]:
#separating notes into respective categories based on keywords or format (for dates)
keyword_to_column = {
    'station': 'station_info',
    'parent jar': 'parent_jar',
    'gear': 'gear_info',
    'net': 'gear_info',
    'formalin': 'formalin',
    'lid': 'curation_notes',
    'top up': 'curation_notes',
    'needs': 'curation_notes',
    'dried': 'curation_notes',
    'dry': 'curation_notes',
    'open': 'curation_notes',
    'plastic': 'curation_notes',
    'imaging': 'curation_notes',
    'quality': 'curation_notes',
    'display': 'curation_notes',
    'loose': 'curation_notes',
    'depth': 'depth_info',
    'date': 'date_info',
    'year': 'date_info',
    'root': 'root_id',
    'inside': 'inside_info',
    'within': 'inside_info',
    'label': 'label_info',
    'says': 'label_info',
    'sticker': 'label_info',
    'age': 'age_info',
    'sex': 'sex_info',
    'species': 'taxonomy_info',
    'genus': 'taxonomy_info',
    'families': 'taxonomy_info',
    'family': 'taxonomy_info',
    'order': 'taxonomy_info',
    'time': 'time',
    'GMT': 'time',
    'assigned': 'assigned'
}


final_columns = ['station_info', 'label_info', 'parent_jar', 'gear_info', 'formalin', 'inside_info',
                 'depth_info', 'date_info', 'time', 'age_info', 'sex_info','taxonomy_info', 'root_id', 'assigned', 'curation_notes', 'other_notes']


notes_parsed = pd.DataFrame(columns=final_columns, index=data.index)


def classify_note(note):
    for keyword, column in keyword_to_column.items():
        if keyword in note.lower():
            return column
#for dates and date ranges:
    date_pattern = r'\b\d{1,2}([./-]\d{1,2})?([./-]\d{4})\b'
    date_range_pattern = r'\b\d{1,2}[\u2013\u2014\-–]{1}\d{1,2}[./-]\d{1,2}[./-]\d{4}\b'

    if re.search(date_range_pattern, note.strip()):
        return 'date_info'
    if re.search(date_pattern, note.strip()):
        return 'date_info'

    return 'other_notes'


for idx, note in data['notes'].dropna().items():
    fragments = [frag.strip() for frag in note.split(';') if frag.strip()]
    categorized = {col: [] for col in final_columns}

    for frag in fragments:
        category = classify_note(frag)
        categorized[category].append(frag)


    for col, entries in categorized.items():
        if entries:
            notes_parsed.at[idx, col] = '; '.join(entries)


data_with_notes = pd.concat([data, notes_parsed], axis=1)



### Saving .csv with separated Notes

In [ ]:
#to save version with separated notes
savecsv = data_with_notes.to_csv('data_with_notes.csv', index=False)